**Cleans VCFs into singletons and removes recombination regions 3+ SNPs within 2kb region sliding window
incorporates codons, opportunities, and trinuc context in separate csv, also checks if we can safely collapse into 6 class**

In [5]:
import os
import glob
import csv
import collections
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict

# paths
base_folder   = os.path.expanduser("~/shared-team/2025-masters-project/people/eleanor/original_and_processed_files")
output_folder = os.path.join(base_folder, "processed_c")
os.makedirs(output_folder, exist_ok=True)

BASES      = "ACGT"
COMPLEMENT = {"A": "T", "T": "A", "C": "G", "G": "C"}

# define functions
# Load CDS features from GFF, returning genes list and cds_features list
def load_gff(gff_file):
    genes        = []
    cds_features = []
    with open(gff_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            cols = line.strip().split("\t")
            if len(cols) < 7 or cols[2] != "CDS":
                continue
            chrom  = cols[0]
            start  = int(cols[3])
            end    = int(cols[4])
            strand = cols[6] if cols[6] in ("+", "-") else "+"
            genes.append((chrom, start, end))
            cds_features.append({
                "chrom":  chrom,
                "start":  start,
                "end":    end,
                "strand": strand,
            })
    return genes, cds_features

#   Build set of genic positions from genes list
def build_genic_positions(genes):
    genic_positions = defaultdict(set)
    for chrom, start, end in genes:
        for p in range(start, end + 1):
            genic_positions[chrom].add(p)
    return genic_positions

# define gene starts/ends
def is_in_gene(chrom, pos, genes):
    for g_chrom, start, end in genes:
        if chrom == g_chrom and start <= pos <= end:
            return True
    return False

#count bases in genic and intergenic regions
def count_bases_fasta(fasta_file, genic_positions):
    genic_counts      = {"A": 0, "C": 0, "G": 0, "T": 0}
    intergenic_counts = {"A": 0, "C": 0, "G": 0, "T": 0}
    current_chrom     = None
    current_pos       = 0

    with open(fasta_file) as f:
        for line in f:
            if line.startswith(">"):
                current_chrom = line.strip().lstrip(">").split()[0]
                current_pos   = 1
                continue
            for b in line.strip().upper():
                if b in genic_counts:
                    if current_pos in genic_positions[current_chrom]:
                        genic_counts[b] += 1
                    else:
                        intergenic_counts[b] += 1
                current_pos += 1

    return genic_counts, intergenic_counts

# load FASTA file into sequence dictionary
def load_fasta_dict(fasta_file):
    """Load FASTA file into {chrom: sequence} dictionary."""
    fasta_dict    = {}
    current_chrom = None
    seq_parts     = []
    with open(fasta_file) as f:
        for line in f:
            if line.startswith(">"):
                if current_chrom is not None:
                    fasta_dict[current_chrom] = "".join(seq_parts).upper()
                current_chrom = line.strip().lstrip(">").split()[0]
                seq_parts     = []
            else:
                seq_parts.append(line.strip().upper())
        if current_chrom is not None:
            fasta_dict[current_chrom] = "".join(seq_parts).upper()
    return fasta_dict

#count trinucleotide opp in genic and intergenic regions collapsed to pyrimidine reference
def count_trinucleotide_opportunities(fasta_dict, genic_positions):
    genic_trinuc      = defaultdict(int)
    intergenic_trinuc = defaultdict(int)

    for chrom, seq in fasta_dict.items():
        for idx in range(1, len(seq) - 1):
            ref = seq[idx]
            if ref not in BASES:
                continue
            five_prime  = seq[idx - 1]
            three_prime = seq[idx + 1]
            if any(b not in BASES for b in [five_prime, three_prime]):
                continue
            if ref in "AG":
                five_prime  = COMPLEMENT[three_prime]
                three_prime = COMPLEMENT[seq[idx - 1]]
                ref         = COMPLEMENT[ref]
            trinuc_simple = f"{five_prime}{ref}{three_prime}"
            pos = idx + 1  # convert to 1-based
            if pos in genic_positions[chrom]:
                genic_trinuc[trinuc_simple] += 1
            else:
                intergenic_trinuc[trinuc_simple] += 1

    return genic_trinuc, intergenic_trinuc

# Get trinucleotide context for a mutations that occured
def get_trinucleotide(chrom, pos, ref, alt, fasta_dict):
    if chrom not in fasta_dict:
        return None, None, None
    seq = fasta_dict[chrom]
    idx = pos - 1
    if idx < 1 or idx >= len(seq) - 1:
        return None, None, None
    five_prime  = seq[idx - 1]
    three_prime = seq[idx + 1]
    if any(b not in BASES for b in [five_prime, ref, three_prime, alt]):
        return None, None, None
    if ref in "AG":
        orig_five   = seq[idx - 1]
        orig_three  = seq[idx + 1]
        five_prime  = COMPLEMENT[orig_three]
        three_prime = COMPLEMENT[orig_five]
        ref         = COMPLEMENT[ref]
        alt         = COMPLEMENT[alt]
    trinuc        = f"{five_prime}[{ref}>{alt}]{three_prime}"
    trinuc_simple = f"{five_prime}{ref}{three_prime}"
    return trinuc, trinuc_simple, ref

# collapse mutations anf find from vcfs
def get_mutation_class(ref, alt):
    ref, alt = ref.upper().strip(), alt.upper().strip()
    if len(ref) != 1 or len(alt) != 1 or ref == alt:
        return None, None
    if ref not in BASES or alt not in BASES:
        return None, None
    label_12 = f"{ref}->{alt}"
    label_6  = label_12 if ref in "CT" else f"{COMPLEMENT[ref]}->{COMPLEMENT[alt]}"
    return label_12, label_6

# find where cds are
def build_codon_pos_map(cds_features):
    codon_map = defaultdict(dict)
    for feat in cds_features:
        chrom, start, end, strand = feat["chrom"], feat["start"], feat["end"], feat["strand"]
        for pos in range(start, end + 1):
            offset    = (pos - start) % 3 if strand == "+" else (end - pos) % 3
            codon_pos = offset + 1
            existing  = codon_map[chrom].get(pos)
            if existing is None or codon_pos < existing:
                codon_map[chrom][pos] = codon_pos
    return codon_map

# find codon position or integernic
def get_codon_label(chrom, pos, codon_map):
    cp = codon_map.get(chrom, {}).get(pos)
    return "intergenic" if cp is None else f"codon_pos_{cp}"

##################
#main loop, define output files for each species
vcf_files      = glob.glob(os.path.join(base_folder, "*.vcf"))
summary_rows   = []
trinuc_rows    = []
species_counts = collections.defaultdict(lambda: collections.Counter())

print(f"Found {len(vcf_files)} VCF files")

for vcf_file in vcf_files:
    species_name = os.path.basename(vcf_file).replace(".vcf", "")
    print(f"\nProcessing {species_name}")

    singles_file          = os.path.join(output_folder, f"{species_name}_singles.vcf")
    singles_filtered_file = os.path.join(output_folder, f"{species_name}_singles_filtered.vcf")

    #find singletons, keep only those with PASS, write out header lines to file, only count those with ==1
    positions_per_genome = []

    with open(vcf_file) as vcf_in, open(singles_file, "w") as vcf_out:
        for line in vcf_in:
            if line.startswith("#CHROM"):
                header               = line.strip().split("\t")
                samples              = header[9:]
                positions_per_genome = [[] for _ in samples]
                seen_positions       = [set() for _ in samples]
                vcf_out.write(line)
                continue
            elif line.startswith("#"):
                vcf_out.write(line)
                continue

            cols  = line.strip().split("\t")
            chrom, pos, ref, alt = cols[0], int(cols[1]), cols[3], cols[4]

            if cols[6] != "PASS":
                continue
            if ref not in BASES or alt not in BASES or "," in alt or "," in ref:
                continue

            allele_count    = 0
            singleton_index = None
            for i, gt in enumerate(cols[9:]):
                if gt.split(":")[0] in ("1", "1/1", "0/1", "1|0", "0|1"):
                    allele_count += 1
                    singleton_index = i

            if allele_count == 1:
                vcf_out.write(line)
                if (chrom, pos) not in seen_positions[singleton_index]:
                    positions_per_genome[singleton_index].append((chrom, pos))
                    seen_positions[singleton_index].add((chrom, pos))

    print(f"  Rows after singleton filter: {sum(len(g) for g in positions_per_genome)}")

    # recombination filtering using 2kb windows and 3 SNP threshold, remove from file
    window_size        = 2000
    cluster_threshold  = 3
    variants_to_remove = set()

    for genome_positions in positions_per_genome:
        chrom_dict = defaultdict(list)
        for chrom, pos in genome_positions:
            chrom_dict[chrom].append(pos)

        for chrom, pos_list in chrom_dict.items():
            pos_list.sort()
            left = 0
            for right in range(len(pos_list)):
                while pos_list[right] - pos_list[left] > window_size:
                    left += 1
                if right - left + 1 >= cluster_threshold:
                    for k in range(left, right + 1):
                        variants_to_remove.add((chrom, pos_list[k]))

    print(f"  Recombination positions to remove: {len(variants_to_remove)}")

    rows_written = 0
    with open(singles_file) as vcf_in, open(singles_filtered_file, "w") as vcf_out:
        for line in vcf_in:
            if line.startswith("#"):
                vcf_out.write(line)
                continue
            cols  = line.strip().split("\t")
            chrom, pos = cols[0], int(cols[1])
            if (chrom, pos) not in variants_to_remove:
                vcf_out.write(line)
                rows_written += 1

    print(f"  Rows after recombination removal: {rows_written}")

    # load gff and fna files, run functions as defined above
    gff_file   = os.path.join(base_folder, f"{species_name}.gff")
    fasta_file = os.path.join(base_folder, f"{species_name}.fna")

    genes, cds_features = load_gff(gff_file)
    genic_positions     = build_genic_positions(genes)
    fasta_dict          = load_fasta_dict(fasta_file)

    genic_counts, intergenic_counts             = count_bases_fasta(fasta_file, genic_positions)
    genic_trinuc_opp, intergenic_trinuc_opp     = count_trinucleotide_opportunities(fasta_dict, genic_positions)

    codon_map = build_codon_pos_map(cds_features)

    # for each singleton get mutation for 6, 12, intergenic status and codon position
    mutation_counts = defaultdict(int)

    with open(singles_filtered_file) as vcf:
        for line in vcf:
            if line.startswith("#"):
                continue
            cols  = line.strip().split("\t")
            chrom, pos = cols[0], int(cols[1])
            ref, alt   = cols[3], cols[4]

            if ref not in BASES or alt not in BASES:
                continue

            count = sum(
                1 for gt in cols[9:]
                if gt.split(":")[0] in ("1", "1/1", "0/1", "1|0", "0|1")
            )

            if count == 1:
                _, label_6    = get_mutation_class(ref, alt)
                label_12, _   = get_mutation_class(ref, alt)
                is_intergenic = not is_in_gene(chrom, pos, genes)
                codon_label   = get_codon_label(chrom, pos, codon_map)

                mutation_counts[(species_name, label_6, is_intergenic, codon_label)] += 1

                if label_12:
                    species_counts[species_name][label_12] += 1

                trinuc, trinuc_simple, _ = get_trinucleotide(chrom, pos, ref, alt, fasta_dict)
                if trinuc is not None:
                    opp_dict = intergenic_trinuc_opp if is_intergenic else genic_trinuc_opp
                    trinuc_rows.append([
                        species_name, chrom, pos, ref, alt,
                        label_6, trinuc, trinuc_simple,
                        is_intergenic, opp_dict.get(trinuc_simple, 0)
                    ])

    # combine information above into summary, count opportunties and add to file
    for (species, mut_class, intergenic, codon_label), count in mutation_counts.items():
        ref_base = mut_class.split("->")[0]
        counts   = intergenic_counts if intergenic else genic_counts
        if ref_base == "C":
            opportunities = counts["C"] + counts["G"]
        elif ref_base == "T":
            opportunities = counts["T"] + counts["A"]
        else:
            opportunities = 0
        summary_rows.append([species, count, mut_class, intergenic, codon_label, opportunities])


############
#strand symmetry test to see if its justified to collapse into 6 class
sym_records = []
for species, counter in species_counts.items():
    for label_12, n in counter.items():
        _, label_6 = get_mutation_class(*label_12.split("->"))
        sym_records.append({
            "species":           species,
            "mutation_class_12": label_12,
            "mutation_class_6":  label_6,
            "no_mutations":      n
        })

sym_df = pd.DataFrame(sym_records)

pairs = {}
for label_6, grp in sym_df.groupby("mutation_class_6"):
    classes = grp["mutation_class_12"].unique().tolist()
    if len(classes) == 2:
        pairs[label_6] = tuple(sorted(classes))

#loops through species and finds forward/reverse
sym_results = []
for species in sorted(species_counts.keys()):
    sp_df = sym_df[sym_df["species"] == species]

    for label_6, (fwd, rev) in sorted(pairs.items()):
        fwd_n = int(sp_df[sp_df["mutation_class_12"] == fwd]["no_mutations"].sum())
        rev_n = int(sp_df[sp_df["mutation_class_12"] == rev]["no_mutations"].sum())
        total = fwd_n + rev_n

        if total == 0:
            continue
        
        #run binomial test for each pair
        p_val = binomtest(fwd_n, total, p=0.5, alternative="two-sided").pvalue
        prop  = fwd_n / total

        sym_results.append({
            "species":         species,
            "pair":            f"{fwd}/{rev}",
            "collapsed_class": label_6,
            "fwd":             fwd,
            "rev":             rev,
            "fwd_count":       fwd_n,
            "rev_count":       rev_n,
            "total":           total,
            "fwd_proportion":  round(prop, 3),
            "p_value":         p_val,
            "significant":     p_val < 0.05,
            "verdict":         "ASYMMETRIC" if p_val < 0.05 else "symmetric"
        })


#bonferroni adjustment
sym_results_df = pd.DataFrame(sym_results)
_, p_adj, _, _  = multipletests(sym_results_df["p_value"], method="bonferroni")
sym_results_df["p_adjusted"]                  = p_adj
sym_results_df["significant_after_correction"] = p_adj < 0.05

#save
sym_results_df.to_csv(
    os.path.join(output_folder, "species_strand_symmetry.csv"), index=False)
print(f"\nStrand symmetry results written to species_strand_symmetry.csv")


# save files
output_csv = os.path.join(output_folder, "mutation_summary_final_c.csv")
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["species", "no_mutations", "mutation_class",
                     "is_intergenic", "codon_position", "opportunities"])
    writer.writerows(summary_rows)
print(f"Mutation summary written to {output_csv}")

trinuc_csv = os.path.join(output_folder, "trinucleotide_context.csv")
trinuc_raw = pd.DataFrame(trinuc_rows, columns=[
    "species", "chrom", "pos", "ref", "alt",
    "mutation_class", "trinucleotide", "trinucleotide_simple",
    "is_intergenic", "opportunities"
])
trinuc_agg = (
    trinuc_raw
    .groupby(["species", "trinucleotide", "trinucleotide_simple",
              "mutation_class", "is_intergenic"])
    .agg(
        mutation_count=("trinucleotide", "count"),
        opportunities=("opportunities", "first")
    )
    .reset_index()
)
trinuc_agg.to_csv(trinuc_csv, index=False)
print(f"Trinucleotide context table written to {trinuc_csv}")


Found 14 VCF files

Processing paeruginosa
  Rows after singleton filter: 36707
  Recombination positions to remove: 19450
  Rows after recombination removal: 17257

Processing styphimurium
  Rows after singleton filter: 4250
  Recombination positions to remove: 598
  Rows after recombination removal: 3652

Processing bpertussis
  Rows after singleton filter: 3172
  Recombination positions to remove: 924
  Rows after recombination removal: 2248

Processing sagalactiae
  Rows after singleton filter: 12093
  Recombination positions to remove: 1498
  Rows after recombination removal: 10595

Processing nmeningitidis
  Rows after singleton filter: 23532
  Recombination positions to remove: 18919
  Rows after recombination removal: 4613

Processing cjejuni
  Rows after singleton filter: 14356
  Recombination positions to remove: 8207
  Rows after recombination removal: 6149

Processing ecoli
  Rows after singleton filter: 44899
  Recombination positions to remove: 11963
  Rows after recombin